[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc5_abtest/corrections/seance1_correction.ipynb)

# Séance 5.1 — Causalité et A/B testing — mesurer ce qu'une campagne fait vraiment

**Correction** · durée : 6h (2h de cours, 2h d'étude de cas, 2h de correction)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer une question prédictive d'une question causale
- nommer le contrefactuel, l'ATE et l'ATT, et dire pourquoi on ne les observe jamais
- décomposer une comparaison de moyennes en effet causal + biais de sélection
- vérifier qu'un tirage au sort a fonctionné avec un tableau d'équilibre
- chiffrer l'effet d'un A/B test, son incertitude, et le traduire en décision

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
data = pd.read_csv(BASE + "hillstrom.csv")
AUCUN, HOMME, FEMME = "No E-Mail", "Mens E-Mail", "Womens E-Mail"


def comparer(a, b):
    """Effet, intervalle a 95 % et p-value entre deux groupes."""
    effet = a.mean() - b.mean()
    es = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    return pd.Series({"effet": effet, "bas_95": effet - 1.96 * es,
                      "haut_95": effet + 1.96 * es, "p_value": p})


print(data.shape, "|", data["segment"].unique())

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Première inspection

> **Votre mission :**
> - Combien de clients le fichier contient-il ? → `n_lignes`
> - Combien de valeurs manquantes en tout ? → `n_manquants`
> - Combien de clients ont reçu l'email hommes ? → `n_homme`

In [ ]:
# .shape renvoie (lignes, colonnes) : le [0] prend les lignes
n_lignes = data.shape[0]

# .isna().sum() donne un total PAR COLONNE : il faut resommer pour le total
n_manquants = data.isna().sum().sum()
n_homme = data["segment"].value_counts()[HOMME]

print(n_lignes, "clients |", n_manquants, "valeurs manquantes |", n_homme, "traites")

In [ ]:
verifier("1a - nombre de clients", n_lignes == 64000, "data.shape[0]")
verifier("1b - valeurs manquantes", n_manquants == 0, "sommez deux fois")
verifier("1c - clients traites", n_homme == 21307, "value_counts() sur la colonne segment")

### Exercice 2 — Un monde sans tirage au sort

> **Votre mission :**
> - Avant d'analyser la vraie expérience, simulons le monde où l'entreprise **n'aurait pas** randomisé : comme beaucoup, elle aurait écrit à ses **meilleurs clients**.
> - On garde les clients « email hommes » dont la dépense passée dépasse la médiane, et les clients « aucun email » en dessous.
> - Compléter la médiane, puis compter les clients de cette base → `n_monde`

In [ ]:
mediane = data["history"].median()

# Les gros clients recoivent l'email, les petits non : c'est l'histoire des
# tablettes dans les ecoles riches, version marketing
monde = pd.concat([
    data.query("segment == @HOMME and history > @mediane"),
    data.query("segment == @AUCUN and history <= @mediane"),
])
monde["email"] = (monde["segment"] == HOMME).astype(int)

n_monde = len(monde)
print("mediane :", mediane, "| base simulee :", n_monde, "clients")

In [ ]:
verifier("2 - taille du monde parallele", n_monde == 21333,
         "la mediane coupe l'echantillon en deux, pas le quartile")

### Exercice 3 — L'estimation naïve

> **Votre mission :**
> - Dans ce monde parallèle, calculer la dépense moyenne des clients avec email et sans email, puis leur différence → `naif` (arrondie à 2 décimales).
> - **Notez ce chiffre quelque part.** C'est ce qu'un analyste pressé appellerait « l'effet de l'email ». Nous le confronterons à la vérité à l'exercice 8.

In [ ]:
moyennes = monde.groupby("email")["spend"].mean()
naif = round(moyennes[1] - moyennes[0], 2)

print(moyennes.round(3))
print("estimation naive de « l'effet de l'email » :", naif, "$")

In [ ]:
verifier("3 - estimation naive", naif == 1.32,
         "moyenne des email=1 moins moyenne des email=0, arrondie a 2 decimales")

### Exercice 4 — Diagnostiquer le biais

> **Votre mission :**
> - Toujours dans le monde parallèle : comparer la **dépense passée** (`history`) des deux groupes → `passe_email` et `passe_sans` (arrondies à 1 décimale).
> - Ces deux clients auraient-ils dépensé pareil **sans aucun email** ?

In [ ]:
passe = monde.groupby("email")["history"].mean().round(1)

passe_email = passe[1]
passe_sans = passe[0]
print("depense passee :", passe_email, "$ avec email contre", passe_sans, "$ sans")

# 414 $ contre 74 $ : ces deux groupes n'ont rien de comparable. Les premiers
# auraient depense davantage MEME SANS EMAIL, donc E[Y0|T=1] > E[Y0|T=0].
# Le biais de selection est positif : la comparaison naive additionne l'effet
# de l'email et l'ecart pre-existant entre bons et mauvais clients.

In [ ]:
verifier("4a - depense passee, groupe traite", passe_email == 414.0, "arrondissez a 1 decimale")
verifier("4b - depense passee, groupe temoin", passe_sans == 73.7, "groupby sur email, colonne history")

### Exercice 5 — Le test d'équilibre

> **Votre mission :**
> - Retour à la **vraie** expérience. Si le tirage au sort a fonctionné, les variables mesurées **avant** l'envoi doivent être quasi identiques d'un groupe à l'autre.
> - Construire le tableau d'équilibre, puis mesurer l'écart maximal de dépense passée entre les trois groupes → `ecart_passe` (arrondi à 2 décimales).

In [ ]:
equilibre = data.groupby("segment").agg(
    clients=("segment", "size"),
    recence=("recency", "mean"),
    passe=("history", "mean"),
    nouveaux=("newbie", "mean"),
)
ecart_passe = round(equilibre["passe"].max() - equilibre["passe"].min(), 2)

# 1,95 $ d'ecart sur ~242 $ de depense passee, soit moins de 1 % : les trois
# groupes sont indiscernables avant l'envoi. Le tirage au sort a fonctionne.
print("ecart maximal sur la depense passee :", ecart_passe, "$")
equilibre.round(3)

In [ ]:
verifier("5 - ecart sur la depense passee", ecart_passe == 1.95,
         "le maximum de la colonne passe moins son minimum")

### Exercice 6 — Les résultats de l'expérience

> **Votre mission :**
> - Pour chaque groupe : le nombre de clients, le taux de visite, le taux d'achat et la dépense moyenne.
> - Récupérer le taux d'achat du groupe « email hommes » → `achat_homme` (arrondi à 4 décimales).
> - *Rappel :* pour une variable qui vaut 0 ou 1, la **moyenne est la proportion de 1**.

In [ ]:
resultats = data.groupby("segment").agg(
    clients=("segment", "size"),
    visite=("visit", "mean"),
    achat=("conversion", "mean"),
    depense=("spend", "mean"),
)
# On garde `resultats` NON arrondi : les effets de l'exercice 7 se calculent
# dessus, et arrondir avant de soustraire fausserait l'effet relatif
achat_homme = round(resultats.loc[HOMME, "achat"], 4)

print("taux d'achat, email hommes :", achat_homme)
resultats.round(4)

In [ ]:
verifier("6 - taux d'achat du groupe traite", achat_homme == 0.0125,
         "la moyenne d'une colonne 0/1 est sa proportion de 1")

### Exercice 7 — Effet absolu, effet relatif

> **Votre mission :**
> - Mesurer l'effet de l'email hommes sur le taux d'achat, par rapport au groupe sans email.
> - En **points de pourcentage** → `abs_conv` (2 décimales), et en **pourcentage du niveau de départ** → `rel_conv` (1 décimale).
> - Les deux décrivent le même résultat. Laquelle des deux mettriez-vous sur une slide ?

In [ ]:
ecart_achat = resultats.loc[HOMME, "achat"] - resultats.loc[AUCUN, "achat"]

abs_conv = round(100 * ecart_achat, 2)
rel_conv = round(100 * ecart_achat / resultats.loc[AUCUN, "achat"], 1)

print("effet absolu :", abs_conv, "points | effet relatif :", rel_conv, "%")

# Le meme fait, deux habillages : « +0,68 point » ou « +119 % ». Quand le
# niveau de depart est minuscule (0,57 %), le relatif est spectaculaire sans
# rien exagerer. Une slide honnete donne les DEUX, ou l'absolu avec son point
# de depart : « de 0,57 % a 1,25 % ».

In [ ]:
verifier("7a - effet absolu", abs_conv == 0.68, "en points : multipliez l'ecart par 100")
verifier("7b - effet relatif", rel_conv == 118.8,
         "divisez l'ecart par le taux du groupe temoin, pas par 1")

### Exercice 8 — Le moment de vérité

> **Votre mission :**
> - Mesurer l'effet **réel** de l'email hommes sur la dépense → `effet_dep` (2 décimales).
> - Le comparer à votre estimation naïve de l'exercice 3 : de combien le monde parallèle se trompait-il ? → `biais` (2 décimales)

In [ ]:
effet_dep = round(resultats.loc[HOMME, "depense"] - resultats.loc[AUCUN, "depense"], 2)
biais = round(naif - effet_dep, 2)

print("effet reel (randomise) :", effet_dep, "$")
print("estimation naive (ex. 3) :", naif, "$")
print("biais de selection :", biais, "$")

# 1,32 contre 0,77 : le monde parallele surestime l'effet de 0,55 $ par
# client, soit un facteur 1,7. Ce 0,55 $, c'est le biais de selection —
# exprime en dollars. Sur 100 000 clients, c'est 55 000 $ de ROI imaginaire.

In [ ]:
verifier("8a - effet reel sur la depense", effet_dep == 0.77,
         "difference des colonnes depense entre HOMME et AUCUN")
verifier("8b - taille du biais", biais == 0.55, "l'estimation naive moins l'effet reel")

### Exercice 9 — Est-ce que ça peut être le hasard ?

> **Votre mission :**
> - Même avec un tirage au sort parfait, deux groupes ne sont jamais exactement identiques. L'effet mesuré dépasse-t-il ce que le hasard seul produirait ?
> - Utiliser `comparer(...)` sur la dépense, email hommes contre aucun email.
> - Relever les bornes de l'intervalle à 95 % → `bas` et `haut` (2 décimales).

In [ ]:
dep_homme = data.query("segment == @HOMME")["spend"]
dep_aucun = data.query("segment == @AUCUN")["spend"]

test_dep = comparer(dep_homme, dep_aucun)
bas = round(test_dep["bas_95"], 2)
haut = round(test_dep["haut_95"], 2)

# L'intervalle va de 0,49 a 1,05 $ : il ne contient pas zero. Meme dans le
# scenario le plus pessimiste compatible avec nos donnees, l'email rapporte.
print("intervalle a 95 % : de", bas, "a", haut, "$")
test_dep.round(4)

In [ ]:
verifier("9a - borne basse", bas == 0.49, "test_dep['bas_95'], arrondi a 2 decimales")
verifier("9b - borne haute", haut == 1.05, "test_dep['haut_95']")
verifier("9c - l'intervalle exclut zero", bas > 0, "regardez le signe de la borne basse")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - Calculer l'effet relatif de l'email hommes sur la **dépense** → `rel_dep` (1 décimale).
> - Puis rédigez en commentaire, en trois phrases maximum, ce que vous diriez à la directrice marketing.

In [ ]:
rel_dep = round(100 * effet_dep / resultats.loc[AUCUN, "depense"], 1)

print("effet relatif sur la depense :", rel_dep, "%")

# Recommandation possible :
# "L'email produits hommes fait passer le taux d'achat de 0,57 % a 1,25 % et
#  augmente la depense de 0,77 $ par client contacte, soit +118 %. Comme les
#  trois groupes ont ete tires au sort, cet ecart est bien un effet de
#  l'email, et non un effet de la qualite des clients cibles. Reste a verifier
#  qu'il couvre le cout d'envoi : c'est la question suivante."

In [ ]:
verifier("10 - effet relatif sur la depense", rel_dep == 118.0,
         "l'effet divise par la depense moyenne du groupe temoin")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 11 — Traduire dans le langage du cours

> **Votre mission :**
> - Répondez en une phrase par question, sans coder.
> - 1. Ici, qu'est-ce que $T_i$ ? (Attention : il y a **deux** traitements possibles — on comparera chacun au groupe sans email.)
> - 2. Si on s'intéresse aux dépenses, qu'est-ce que $Y_i$ ?
> - 3. Le client n° 42 a reçu l'email hommes et a dépensé 0 $. Que représenterait son $Y_{0,42}$ ? Peut-on l'observer ?
> - 4. Comment s'appelle, dans le cours, ce résultat qu'on ne peut jamais observer ?

**Corrigé.**

1. $T_i$ vaut 1 si le client $i$ a reçu un email, 0 sinon. Comme il y a deux
   emails différents, on définit en pratique **deux comparaisons**, chacune
   avec son $T_i$ binaire : *hommes vs aucun*, puis *femmes vs aucun*.

2. $Y_i$ est la colonne `spend` : la dépense observée du client $i$ pendant
   les deux semaines qui ont suivi l'envoi.

3. $Y_{0,42}$ est ce que le client 42 **aurait dépensé s'il n'avait reçu aucun
   email**, tout le reste étant identique. On ne peut **pas** l'observer : il a
   reçu l'email, donc on ne voit que $Y_{1,42} = 0$.

4. C'est le **contrefactuel**. C'est le problème fondamental de l'inférence
   causale : pour un individu donné, on n'observe jamais les deux résultats
   potentiels.

### Question 12 — Prédiction ou causalité ?

> **Votre mission :**
> - Pour chaque question, dire s'il s'agit d'une question **prédictive** (bloc 4) ou **causale** (ce bloc), et justifier en une phrase.
> - 1. Quels clients ont le plus de chances d'acheter dans les deux prochaines semaines ?
> - 2. Envoyer un email augmente-t-il la probabilité d'achat ?
> - 3. Combien ce client va-t-il probablement dépenser ?
> - 4. Que dépenseraient nos clients si nous leur envoyions l'email hommes plutôt que rien ?

**Corrigé.**

1. **Prédictive.** On veut classer les clients par probabilité d'achat. Peu
   importe *pourquoi* ils achètent : seule la qualité du classement compte.
2. **Causale.** « Augmente-t-il » désigne une intervention : il faut comparer
   deux résultats potentiels, avec et sans email.
3. **Prédictive.** On estime un montant probable, sans rien changer au monde.
4. **Causale.** C'est littéralement une question contrefactuelle : « que
   *seraient* les dépenses *si* nous faisions X ».

Le test qui tranche : **la question contient-elle un « si on faisait X » ?**

Un excellent modèle prédictif répond parfaitement à 1 et 3 et se trompe
complètement sur 2 et 4 — c'est l'exemple des prix d'hôtels du cours.

### Question 13 — Le détecteur de randomisation cassée

> **Votre mission :**
> - Refaire le tableau d'équilibre de l'exercice 5, mais sur `monde` (en groupant par `email`).
> - Qu'est-ce qui saute aux yeux ? Si on vous avait livré cette base en vous affirmant qu'elle était randomisée, ce test vous aurait-il sauvé ?

In [ ]:
equilibre_monde = monde.groupby("email").agg(
    clients=("email", "size"),
    recence=("recency", "mean"),
    passe=("history", "mean"),
    nouveaux=("newbie", "mean"),
)
equilibre_monde.round(3)

# 414 $ contre 74 $ de depense passee : l'ecart creve l'ecran, la ou la vraie
# experience tenait dans 1,95 $. La recence bouge aussi (5,1 contre 6,3),
# parce qu'elle est correlee a `history` — le biais contamine tout ce qui
# touche a la qualite du client.
#
# Oui, ce test aurait sauve : c'est un DETECTEUR de randomisation cassee. Il
# ne demande ni le protocole ni la confiance de personne, seulement les
# donnees. Dans la vraie vie (bug de ciblage, desabonnements, doublons), il
# echoue plus souvent qu'on ne croit.

### Question 14 — Trois graphiques, trois indicateurs

> **Votre mission :**
> - Faire un graphique en barres horizontales par indicateur, à partir du tableau `resultats` : taux de visite, taux d'achat, dépense moyenne.
> - Trier avant de tracer, et une idée par figure.
> - *Rappel :* `resultats["visite"].sort_values().plot(kind="barh", figsize=(7, 3))`

In [ ]:
for colonne, titre in [("visite", "Taux de visite"),
                       ("achat", "Taux d'achat"),
                       ("depense", "Depense moyenne ($)")]:
    resultats[colonne].sort_values().plot(kind="barh", figsize=(7, 3),
                                          color="steelblue")
    plt.title(titre)
    plt.ylabel("")
    plt.show()

# Les trois figures racontent la meme histoire dans le meme ordre : aucun
# email < email femmes < email hommes. Attention a l'axe : il part de zero
# ici, et c'est ce qui evite de faire passer +0,7 point pour un raz-de-maree.

### Question 15 — Quelle campagne est la meilleure ?

> **Votre mission :**
> - Attention au raccourci : « A bat le témoin, B bat le témoin, et l'effet de A est plus grand, donc A bat B ». Pour affirmer que A bat B, il faut **comparer A à B directement**.
> - Comparer l'email hommes à l'email femmes sur le taux d'achat, puis sur la dépense.
> - La supériorité de l'un sur l'autre est-elle établie dans les **deux** cas ?

In [ ]:
duel = pd.DataFrame({
    "achat": comparer(data.query("segment == @HOMME")["conversion"],
                      data.query("segment == @FEMME")["conversion"]),
    "depense": comparer(data.query("segment == @HOMME")["spend"],
                        data.query("segment == @FEMME")["spend"]),
}).T
duel.round(4)

# Sur l'ACHAT : +0,37 point, intervalle [0,0017 ; 0,0056], p = 0,0002.
# L'email hommes bat nettement l'email femmes.
#
# Sur la DEPENSE : +0,35 $, mais l'intervalle [0,033 ; 0,658] frole zero et
# p = 0,030. L'ecart passe le seuil, de justesse — la borne basse dit qu'un
# avantage de 3 centimes reste compatible avec nos donnees.
#
# Conclusion honnete : l'email hommes est meilleur sur l'achat, et
# probablement meilleur sur la depense, sans que ce soit tranche.

### Question 16 — Le petit concurrent

> **Votre mission :**
> - Un concurrent plus petit n'a que **2 000 clients** pour mener le même test. Simuler sa situation avec `data.sample(2000, random_state=0)`, puis comparer l'email hommes au groupe sans email sur le taux d'achat.
> - Recommencer avec plusieurs valeurs de `random_state`. Que constatez-vous sur la largeur de l'intervalle et sur la stabilité de l'effet estimé ?
> - L'effet de l'email a-t-il « disparu » chez lui ? Que peut-il conclure — et surtout, que ne peut-il **pas** conclure ?

In [ ]:
for graine in range(5):
    petit = data.sample(2000, random_state=graine)
    r = comparer(petit.query("segment == @HOMME")["conversion"],
                 petit.query("segment == @AUCUN")["conversion"])
    largeur = r["haut_95"] - r["bas_95"]
    print(f"graine {graine} : effet {r['effet']:+.4f} | largeur {largeur:.4f}"
          f" | p {r['p_value']:.3f}")

print("largeur sur les 64 000 clients : 0.0036")

# L'intervalle passe de 0,0036 a environ 0,020 : cinq a six fois plus large.
# C'est attendu, l'erreur standard varie en 1/racine(n) et racine(32) = 5,7.
# La p-value depasse presque toujours 0,05, et l'effet estime saute d'un
# tirage a l'autre (de +0,004 a +0,008 selon la graine).
#
# Non, l'effet n'a pas disparu : c'est le meme monde, le meme email. Le
# concurrent peut seulement conclure que SON TEST N'A PAS LA PRECISION
# necessaire pour detecter un effet de cette taille. Il ne peut surtout pas
# conclure que l'email est inefficace.

### Question 17 — Est-ce que ça rapporte ?

> **Votre mission :**
> - La directrice donne ses paramètres : l'entreprise peut contacter **100 000 clients**, la marge est de **40 %** du chiffre d'affaires, et chaque email coûte **0,05 $**.
> - Bénéfice net par client = effet sur la dépense × taux de marge − coût de l'email.
> - Calculer le bénéfice attendu sur 100 000 clients pour chaque campagne. Puis refaire le calcul avec les **bornes** de l'intervalle de confiance : la décision tient-elle dans le scénario le plus prudent ?

In [ ]:
CIBLE, MARGE, COUT = 100_000, 0.40, 0.05

lignes = []
for campagne in [HOMME, FEMME]:
    r = comparer(data.query("segment == @campagne")["spend"], dep_aucun)
    for nom in ["effet", "bas_95", "haut_95"]:
        net = r[nom] * MARGE - COUT
        lignes.append({"campagne": campagne[:5], "scenario": nom,
                       "net_client": round(net, 3),
                       "net_100k": round(net * CIBLE)})

pd.DataFrame(lignes)

# Scenario central : hommes +25 800 $, femmes +12 000 $ sur 100 000 clients.
# Aux bornes : hommes entre +14 400 et +37 200 $, femmes entre +1 800 et
# +22 200 $. Les deux campagnes restent rentables MEME a la borne basse : la
# decision est robuste a l'incertitude statistique, et cet argument-la parle
# infiniment plus a une direction qu'une p-value.
#
# Marge de securite tres differente cependant : l'email femmes ne tient qu'a
# 1 800 $ pres dans le pire cas.

### Question 18 — Segmenter, et se méfier de soi-même

> **Votre mission :**
> - Les campagnes ont-elles le même effet selon le canal d'achat habituel (`channel`) ? Calculer la dépense moyenne par `channel` et par `segment`, puis l'effet de chaque campagne dans chaque canal.
> - Repérer le canal où l'écart semble le plus fort. Recommanderiez-vous de concentrer le budget dessus ?
> - *Nouveau :* `data.pivot_table(values="spend", index="channel", columns="segment", aggfunc="mean")` croise deux variables en un tableau.

In [ ]:
seg = data.pivot_table(values="spend", index="channel",
                       columns="segment", aggfunc="mean")
seg["effet_homme"] = seg[HOMME] - seg[AUCUN]
seg["effet_femme"] = seg[FEMME] - seg[AUCUN]

seg[["effet_homme", "effet_femme"]].round(3)

# Multichannel sort tres au-dessus : +1,21 $ contre +0,56 $ pour Phone. La
# tentation est de tout miser dessus.
#
# Non, pas sur cette base. C'est une analyse EXPLORATOIRE : 3 canaux x 2
# campagnes x plusieurs indicateurs, cela fait beaucoup de comparaisons, et
# plus on decoupe, plus on risque de prendre une fluctuation pour un effet.
# La bonne conclusion n'est pas « ciblons Multichannel » mais « TESTONS
# l'hypothese Multichannel dans une nouvelle experience dediee, avec le KPI
# et les segments fixes a l'avance ».

### Question 19 — Le stagiaire revient

> **Votre mission :**
> - Nouvelle idée du stagiaire : « Comparons la dépense des clients qui ont **visité** le site à celle des autres : on verra l'effet causal de la visite ! »
> - Reproduire son calcul : la dépense moyenne des clients traités **qui ont visité**, contre celle du groupe témoin entier. Retrouvez-vous les chiffres de sa slide ?
> - Regarder aussi ce que dépensent les clients traités qui n'ont **pas** visité.
> - `visit` est-elle une variable mesurée **avant** ou **après** l'envoi ? Pourquoi cette comparaison retombe-t-elle exactement dans le piège du monde parallèle ?

In [ ]:
slide = data.query("segment == @HOMME and visit == 1")["spend"].mean()
sans_visite = data.query("segment == @HOMME and visit == 0")["spend"].mean()

print("traites QUI ONT VISITE :", round(slide, 2), "$")
print("groupe temoin entier   :", round(dep_aucun.mean(), 2), "$")
print("rapport                :", round(slide / dep_aucun.mean(), 1))
print("traites SANS visite    :", sans_visite, "$")

# 7,78 $ contre 0,65 $ : les chiffres de la slide sont exacts, au centime.
# Et le dernier chiffre explique tout : les traites qui n'ont pas visite
# depensent EXACTEMENT 0 $. Forcement — on ne peut pas acheter sans passer
# sur le site. Filtrer sur `visit`, c'est donc garder d'office tous les
# acheteurs et jeter tous les non-acheteurs.
#
# `visit` est POST-TRAITEMENT : elle est en partie causee par l'email. Les
# visiteurs sont auto-selectionnes — plus interesses, plus acheteurs de
# nature — donc leur Y0 n'a rien a voir avec celui du temoin entier. Meme
# structure exacte que le monde parallele : l'ecart melange l'effet de
# l'email et la selection.
#
# Regle d'or : on ne conditionne JAMAIS sur une variable post-traitement.

### Question 20 — La note à la direction

> **Votre mission :**
> - Rédigez votre recommandation en **cinq phrases maximum**. Elle sera jugée sur cette liste :
> - une stratégie claire est recommandée ; l'effet sur l'achat et sur la dépense est cité, en choisissant honnêtement entre absolu et relatif ; l'incertitude est mentionnée (un intervalle, pas seulement « significatif ») ; l'ordre de grandeur du bénéfice apparaît ; au moins une limite est signalée ; **zéro jargon** — la directrice n'a jamais entendu parler de p-value.
> - Puis, en deux lignes chacune : l'expérience date de 2008, sur un site américain — peut-on en garantir les résultats en France en 2026, et comment s'appelle ce problème ? Et : avant de lancer un nouveau test, faut-il fixer le KPI, les segments et la durée à l'avance, ou peut-on attendre les résultats pour choisir ?

**Exemple de note recevable.**

> Nous recommandons de déployer la campagne **email produits hommes** sur les
> 100 000 clients. L'expérience, menée sur 64 000 clients tirés au sort, montre
> qu'elle fait passer le taux d'achat de 0,57 % à 1,25 % et augmente la dépense
> de 0,77 $ par client contacté. La précision du test situe l'effet réel entre
> 0,49 $ et 1,05 $ par client, ce qui garantit la rentabilité même dans le
> scénario le plus prudent. Nous en attendons un bénéfice net de l'ordre de
> 26 000 $, coût d'envoi déduit. Limite principale : le test a duré deux
> semaines — il ne mesure ni la lassitude, ni les désabonnements, ni l'effet
> au-delà de cet horizon.

**Validité dans le temps.** Non, on ne peut rien garantir. C'est le problème
de la **validité externe** : un effet causal est estimé pour *cette*
population, *ce* contexte, *cette* période. Le marché, les habitudes email et
la pression concurrentielle ont changé depuis 2008 — il faudrait retester.

**Ce qu'il faut fixer à l'avance.** Le KPI principal, les segments analysés et
la durée du test, idéalement la taille d'échantillon aussi. Si on choisit
*après* avoir vu les résultats, on finit toujours par trouver un découpage où
« ça marche » par hasard : c'est le problème des comparaisons multiples, et
c'est exactement ce que montrait la question 18.